In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:

!pip install gensim nltk pandas spacy scikit-learn scikit-optimize matplotlib numpy libsvm flask sentence_transformers requests tomotopy stanza classla stanza-batch
!pip install octis --no-dependencies




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.6/170.6 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.1/519.1 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.0/282.0 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.7 MB/s eta 0:00:00

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 8.9 MB/s eta 0:00:00


In [ ]:
import importlib.metadata

desired_version = "1.26.0"

try:
    installed_version = importlib.metadata.version("numpy")
    if installed_version == desired_version:
        print(f"NumPy {desired_version} is already installed.")
    else:
        print(f"Installing NumPy {desired_version} (current: {installed_version})...")
        !pip install numpy=={desired_version} --prefer-binary
        import os
        os._exit(00)  # Restart runtime for changes to take effect
except importlib.metadata.PackageNotFoundError:
    print(f"NumPy is not installed. Installing {desired_version}...")
    !pip install numpy=={desired_version} --prefer-binary
    import os
    os._exit(00)

Installing NumPy 1.26.0 (current: 1.26.4)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 53.8 MB/s eta 0:00:00
^C


In [ ]:
from nltk.corpus import stopwords
import re
import classla
from stanza_batch import batch

excluded_tags = {"NOUN", "VERB", "ADJ", "PROPN"}

classla.download('sl')
class SloPreProcessing():
    def __init__(self) -> None:
        self.nlp = classla.Pipeline(lang="sl", processors="tokenize,pos,lemma")  # Initialize Stanza pipeline
        self.docs = None  # Placeholder for processed documents
        self.stop_words = set(stopwords.words('slovene'))  # Stop words list for Slovenian
    @staticmethod
    def remove_special_characters(text):
        text = text.lower()

        ### REMOVE SPECIAL CHARACTERS ADDEQUATELY!
        # newtext = text.replace("[!@#$%^&*()[]{};:,./<>?\|`~-=_+]", " ") ow l
        # print(newtext)

        return text

    def lemmatize(self, sents_with_lemmas):
        return [word.lemma for sent in sents_with_lemmas.sentences for word in sent.words if word.lemma not in self.stop_words if word.lemma.isalnum() if word.upos in excluded_tags ]

    def pre_process(self, text):
        text = self.remove_special_characters(text)
        return self.lemmatize(text)

    def batch(self, docs):
        lemmatized_coll = []

        for idx, d in enumerate(docs):
            print('doc: ', idx)
            sent = self.nlp(d.lower())
            lemmatized_coll.append(self.lemmatize(sent))

        return lemmatized_coll

    def main(self):
        docs = [
                    "\"Moj dnevni red je zelo preprost. Ima samo tri točke: orožje, orožje, orožje,\" je povedal Kuleba. \"Prepričani smo, da je najboljši način za pomoč Ukrajini, da se ji zagotovi vse potrebno, da ustavijo ruskega predsednika Vladimirja Putina, da se ruska vojska premaga v Ukrajini, na ozemlju Ukrajine, da se vojna ne razširi naprej,\" je pozval.Poudaril je, da sta ukrajinska vojska in ukrajinsko ljudstvo v zadnjih tednih pokazala, da se znata boriti in zmagovati. Vendar pa bodo brez zadostne podpore v obliki vsega orožja, ki ga zahteva Ukrajina, te zmage pospremljene \"z ogromno žrtvami\", je dejal. Povedal je, da potrebujejo letala, protiladijske rakete in težke sisteme zračne obrambe.\"Več orožja dobimo, prej pride v Ukrajino, več človeških življenj bo rešenih. Ne bo več uničenih mest in vasi. In ne bo več ponovitev Buče,\" je povedal Kuleba, ki se je udeležil dopoldanskega dela zasedanja zunanjih ministrov Nata. Članice zveze Nato je pozval, naj opustijo zadržke. \"Ker – kakor nenavadno se sliši – orožje danes služi miru,\" je dodal Kuleba.Zunanji ministri Nata so se medtem na zasedanju v Bruslju strinjali, da je treba okrepiti vojaško pomoč Ukrajini, je povedal generalni sekretar zavezništva Jens Stoltenberg. Zaveznice v Natu so pripravljene storiti več tudi glede vojaške pomoči, je poudaril. Dodal je, da se zavedajo nujnosti pomoči.Zaveznice so se strinjale, da je treba pomagati tudi drugim partnericam, ki jih Rusija ogroža, je poudaril generalni sekretar zavezništva. Pri tem je izpostavil Gruzijo ter Bosno in Hercegovino.Tako Kuleba kot Stoltenberg sta poudarila, da se pri pomoči Ukrajini ne sme razlikovati med defenzivnim in ofenzivnim orožjem. \"Vsako orožje, uporabljeno v rokah ukrajinske vojske na ozemlju Ukrajine proti tujemu napadalcu, je po definiciji defenzivno,\" je povedal ukrajinski zunanji minister. Tiste države, ki pravijo, da bodo zagotavljale defenzivno orožje, ofenzivnega pa ne, je označil za \"hinavske\".Obenem je bil kritičen do Nemčije, za katero je dejal, da bi lahko glede na svoje zmogljivosti storila več za pomoč Ukrajini. Nemška zunanja ministrica Annalena Baerbock je medtem povedala, da Berlin skupaj s partnerji preučuje, kako bi intenzivneje in bolj usklajeno pomagali Ukrajini.Vodja ukrajinske diplomacije je spregovoril tudi o zadnjih predlogih sankcij proti Rusiji. \"Še naprej bomo vztrajali pri popolnem embargu na nafto in plin iz Rusije, izključitvi vseh ruskih bank iz sistema Swift ter zaprtju vseh pristanišč za ruske ladje in rusko blago z nekaj izjemami za humanitarne potrebe,\" je povedal.Izrazil je upanje, da v prihodnje ne bodo potrebna grozodejstva, kakršno se je zgodilo v Buči, da bodo partnerice sprejele nove sankcije.Zasedanja se udeležuje tudi slovenski zunanji minister Anže Logar, ki se je zavzel za čimprejšnjo prepoved uvoza ruskega plina v Evropsko unijo. Zavzel se je tudi za nadaljnjo podporo Ukrajini v njenem boju proti ruski agresiji, tako na humanitarnem kot vojaškem področju.\"Mislimo, da je ta predlog nujen čim prej,\" je Logar dejal glede evropske prepovedi uvoza ruskega plina. \"Kajti dokler bo Evropska unija plačevala krvave energetske evre Rusiji, toliko dlje bo trajala ruska agresija v Ukrajini,\" je poudaril. Dodal je, da \"razvoj dogodkov gre v to smer\". Slovenija ob tem aktivno išče vire za nadomestitev ruskega plina, pri tem pa je po ministrovih besedah zelo uspešna.Države članice EU-ja trenutno razpravljajo o predlogu petega svežnja sankcij, ki ga je ta teden predstavila Evropska komisija. V njem sta med drugim prepoved uvoza premoga iz Rusije in zaprtje evropskih pristanišč za ruske ladje. Po mnenju Logarja to ni zadnji sveženj sankcij proti Rusiji zaradi njene agresije v Ukrajini.Glede podpore zveze Nato, o kateri bodo ministri razpravljali tudi z ukrajinskim kolegom Kulebo, pa je minister dejal, da morajo države članice zavezništva še naprej podpirati Ukrajino v boju za osvoboditev njenega ozemlja v okviru mednarodno priznanih meja, tako na vojaškem kot humanitarnem področju.Slovenija na vseh področjih podpira Ukrajino, je še dejal. Prispevek na področju humanitarne pomoči je po njegovih besedah rekorden, pomagala pa je tudi z ubojnimi sredstvi. Poleg tega Slovenija pomaga državam, ki gostijo begunce iz Ukrajine.Obvestilo uredništva:Zaradi številnih komentarjev in zagotavljanja čim višjih standardov razprave pod članki o dogajanju v Ukrajini smo se odločili, da komentiranje na portalu rtvslo.si omogočimo pod eno novico. Ne gre za cenzuro ali blokado, temveč za vzdrževanje ravni komunikacije na portalu javne RTV, ki je zavezana k takšnim merilom. Svoje mnenje o dogajanju v Ukrajini lahko ob spoštovanju forumskih pravil MMC RTV SLO izrazite v komentarjih tukaj.",
                    "Raketna napada na železniško postajo, s katere civilisti zapuščajo mesto Kramatorsk na vzhodu Ukrajine, naj bi izvedle ruske sile. Pri tem je po zadnjih podatkih umrlo najmanj 40 ljudi, tudi pet otrok, ranjenih je najmanj 87, poroča Guardian. Številni ranjeni so hudo poškodovani. Drugi mediji poročajo o še večjem številu ranjenih.Guverner vzhodne ukrajinske regije Doneck Pavlo Kirilenko je opozoril, da je bilo na postaji med napadom več tisoč ljudi, saj je od tam potekala evakuacija prebivalcev v varnejše pokrajine v Ukrajini.Župan Oleksandr Gončarenko je dejal, da bolnišnice ne zmorejo obravnavati vseh ranjenih.Ukrajinski predsednik Volodimir Zelenski je dejal, da v času napada na postaji ni bilo ukrajinskih enot.Rusko obrambno ministrstvo je sporočilo, da ruska vojska za 8. april v Kramatorsku ni načrtovala nobenih vojaških napadov in da za raketiranje ni odgovorna. Vse izjave ukrajinskih oblasti o raketnem napadu so označili za popolnoma neresnične provokacije. \"Poudarjamo, da taktične rakete točka-U, katerih dele so našli v bližini železniške postaje in so jih objavili očividci, uporabljajo samo ukrajinske oborožene sile,\" je še dodalo ministrstvo.Analitiki medtem opozarjajo, da videoposnetki na družbenih omrežjih kažejo, da naj bi tudi ruske sile uporabljale te rakete, ki so izjemno nenatančne in pogosto zgrešijo tarčo tudi za več kot pol kilometra.Rusko ministrstvo je dodalo, da so ukrajinske sile hotele preprečiti civilistom, da odidejo, da bi jih lahko še naprej uporabljale kot živi ščit. V Moskvi so dodali, ne da bi predstavili dokaze, da so bile rakete izstreljene iz bližnjega mesta Dobropilja, ki je pod nadzorom Ukrajine.Napad je obsodila Evropska unija. \"Grozljivo je videti, da je Rusija napadla eno od glavnih postaj, ki jo civilisti uporabljajo za evakuacijo iz regije, v kateri ruske sile stopnjujejo svoj napad,\" je na Twitterju zapisal predsednik Evropskega sveta Charles Michel.V Ukrajini je tudi tokratna noč minila v znamenju vojnih grozodejstev in bega ljudi z vzhoda in juga države, kjer Rusija začenja obsežno ofenzivo.Tamkajšnje oblasti prebivalce pozivajo, naj zapustijo domove, dokler je to še varno storiti, kajti v prihodnjih tednih pričakujejo obsežno rusko ofenzivo. Ukrajinski zunanji minister Dmitro Kuleba je opozoril, da \"jih tam čakajo tako hudi spopadi kot v drugi svetovni vojni\".Razmere v Borodjanki, kraju severozahodno od Kijeva, ki so ga pred kratkim zapustile ruske sile, so še strašnejše kot v Buči, kjer so ruski vojaki zagrešili poboje civilistov, je zvečer v videonagovoru dejal ukrajinski predsednik Zelenski. \"Žrtev je še več,\" je dodal.V Buči naj bi bilo po podatkih ukrajinskih oblasti ubitih več kot 300 ljudi, 50 med njimi naj bi bilo usmrčenih. Ukrajinska državna tožilka Irina Venediktova je dejala, da so v Borodjanki odkrili 26 trupel pod dvema uničenima zgradbama. Ob tem je izrazila pričakovanje, da bo ruskemu predsedniku Vladimirju Putinu sodilo mednarodno sodišče. Putina je označila za \"največjega vojnega zločinca 21. stoletja\".Moskva sicer vseskozi zanika, da bi bila tarča napadov civilisti. Podobe trupel na cestah v Buči so po njenih besedah uprizorile ukrajinske oblasti, in sicer zato, da bi upravičile še več sankcij proti Rusiji in izpodbijale mirovne pogovore.Borodjanka je od Buče oddaljena okoli 25 kilometrov. Zelenski pa za zdaj svojih trditev ni podkrepil z dokaznim gradivom, da so za pomor odgovorne ruske sile. BBC poroča o tem, da ima dokaze, da so ruski vojaki civiliste na tem območju uporabljali za živi ščit. Pojavil pa se je tudi videoposnetek, na katerem naj bi ukrajinski vojaki ustrelili zajetega ruskega vojaka, kar je, če bo potrjeno, prav tako kršitev ženevskih konvencij.Tiskovni predstavnik Kremlja Dimitrij Peskov je sinoči priznal, da je imela Rusija velike vojaške izgube. Moskva je do zdaj priznala približno 1300 smrtnih žrtev, po ameriških ocenah je število precej večje, od 7000 do 15.000.Rusija in Ukrajina sta pripravljeni nadaljevati pogovore, čeprav so poročila o množičnih pobojih v ukrajinskem mestu Buča upočasnila proces, pa je medtem dejal neimenovani turški uradnik. Dodal je, da nekatera vprašanja ostajajo odprta, med drugim status regij Donbas in Krim ter varnostna jamstva, datum naslednjega kroga pogajanj pa še ni določen.\"Rusija in Ukrajina sta pripravljeni na pogovore v Turčiji, vendar sta daleč od dogovora o skupnem sporazumu,\" je po poročanju francoske tiskovne agencije AFP dejal turški uradnik.Turški zunanji minister Mevlut Cavusoglu je sicer konec marca dejal, da bi se lahko ruski in ukrajinski zunanji minister sešla v dveh tednih, v četrtek pa je priznal, da so poročila o množičnih pobojih civilistov v kraju Buča zasenčila mirovna pogajanja med Rusijo in Ukrajino.Ukrajina je prav tako v četrtek pozvala Rusijo, naj pokaže, da je pripravljena na dialog, tako da zmanjša sovražnost. Moskva pa je ukrajinske pogajalce obtožila, da so po pogajanjih konec marca v Carigradu spremenili svoje zahteve glede pomembnih točk in da ne spoštujejo dogovorov.Ruski zunanji minister Sergej Lavrov je dejal, da je ukrajinska stran v sredo Moskvi predstavila svoj osnutek mirovnega sporazuma, ki po mnenju Rusije vsebuje zanje nesprejemljive elemente. Po besedah Lavrova je namreč Kijev spremenil svoje stališče glede za Rusijo pomembnih točk v povezavi s polotokom Krim, ki si ga je Rusija enostransko priključila leta 2014, in prorusko regijo Doneški bazen na vzhodu Ukrajine.V Kijev je medtem z vlakom prispela Ursula von der Leyen, in sicer v spremstvu z visokim zunanjepolitičnim predstavnikom EU-ja Josepom Borrellom. V Kijevu se bosta sešla z ukrajinskim predsednikom Zelenskim, ki naj bi mu ponudila podporo in zagotovilo glede prošnje Ukrajine za članstvo v EU-ju.V delegaciji so tudi slovaški premier Eduard Heger in več evropskih poslancev.\"Običajno traja več let, preden Svet EU-ja sprejme prošnjo za članstvo, Ukrajina pa je to dosegla v tednu ali dveh. Pozivam, da čim prej nadaljujemo,\" je Ursula von der Leyen dejala glede formalne podpore članstvu Ukrajine v EU-ju. \"Naš cilj je, da Svetu vlogo Ukrajine predstavimo poleti,\" je še dodala.EU bo danes znova odprl svoje predstavništvo v Kijevu, je že na vlaku na poti napovedal Borrell. V Kijev je prispel tudi veleposlanik Matti Maasikas, ki je od septembra 2019 vodil delegacijo EU-ja v Ukrajini. Estonski diplomat bo z majhno ekipo ponovno prevzel naloge v Kijevu. Vrnitev veleposlanika EU-ja naj bi pokazala, \"da Ukrajina obstaja, da obstajajo prestolnica, vlada in predstavništva drugih držav\", je dejal Borrell, ki je o vožnji z vlakom po Ukrajini dejal: \"Ne počutiš se, kot da si v vojni.\"Borell je napovedal tudi, da bo na potovanju predstavil ukrepe, ki jih je EU sprejel za osamitev Rusije zaradi njene invazije v Ukrajini.Borrell je izrazil prepričanje, da se bodo države članice EU-ja strinjale z njegovim predlogom, da se Ukrajini v prihodnjih dneh zagotovi dodatnih 500 milijonov evrov za podporo ukrajinskim oboroženim silam.Iz Kremlja so sporočili, da bo Rusija t. i. posebno operacijo v Ukrajini končala \"v bližnji prihodnosti\", saj da je izvedla vse cilje, ki si jih je zadala tako ruska vojska kot ruski mirovni pogajalci. Tiskovni predstavnik Kremlja Dimitrij Peskov je dejal, da Moskva razume odločitev nekaterih držav, ki so glasovale za izključitev Rusije iz Sveta za človekove pravice ZN-a, saj da so bile v to prisiljene. Med njimi je bila tudi Srbija.Članice EU-ja so medtem potrdile peti sveženj sankcij proti Rusiji zaradi njene invazije v Ukrajini, ki vključuje tudi prepoved uvoza premoga v EU. To bo po navedbah predsednice Evropske komisije Ursule von der Leyen stalo štiri milijarde evrov na leto.Dogovorili so se tudi o prepovedi transakcij na štiri ruske banke, tudi na drugo največjo VTB. Ukrepi vključujejo tudi prepoved izvoza kvantnih računalnikov in naprednih čipov. Nove sankcije so uvedene tudi proti 217 posameznikom in 18 pravnim subjektom.Z ukrepom, vrednim 5,5 milijarde evrov, pa želi EU Rusijo in njene oligarhe odrezati od dobrin, kot so les, cement, morska hrana in alkohol, vključno z vodko.",
                    "Ukrajinci so v grobišču pokopali žrtve, za katere trdijo, da so jih ubile ruske oborožene sile. Vodja regionalne policije v Kijevu Andrij Niebitov je povedal, da je bilo v grobu 40 trupel, med njimi dva pripadnika ukrajinskih vojaških sil, poroča francoska tiskovna agencija AFP.Dodal je, da so na truplih našli strelne rane, kar potrjuje trditve, da so bili umrli namenske tarče napadov in ne slučajne žrtve zračnih napadov ali artilerijskega ognja. \"Te dogodke lahko opredelim kot vojni zločin,\" je dejal.Mesto je obiskala tudi poročevalka RTV Slovenija Karmen Švegl. \"Preiskovalci ravnokar iz grobišča vlečejo trupla. V tem času, ko smo bili tukaj, so jih našteli že 15,\" je dejala. \"Svojci pogrešanih v strahu čakajo, ali bodo morda iz te mokre zemlje potegnili koga izmed njihovih bližnjih.\"Pred tem se je javila z Železniške ulice v mestu, \"kjer so se očitno bíli najhujši spopadi. Videli smo resnično veliko uničenega ruskega orožja, opustošenje, ki so ga pustili za seboj, in še vedno je mogoče videti mesta, na katerih so bili ljudje ubiti – za seboj so pustili vrečke, potovalke, skratka, resnično pretresljivi prizori so tukaj v Buči.\"Bučo pa si je ogledal tudi začasni odpravnik poslov Slovenije v Ukrajini Boštjan Lesjak, ki je prvi tuji diplomat v Buči.Glede prvih vtisov ob ogledu terena je dejal, da so ti \"zelo žalostni, čustveni\". \"Lahko vidimo uničene domove, v katerih so ljudje še pred mesecem dni mirno prebivali. Vidimo lahko, da bo potrebnega veliko časa in sredstev za obnovo, predvsem pa so žalostni pogledi na madeže krvi, kjer so bili pobiti nedolžni ljudje, in pozivam mednarodno javnost, da čim prej pošlje v Bučo mednarodne strokovnjake, ki bi te zločine raziskali. Danes smo videli, da ukrajinska policija izvaja preiskave, dela zapisnike, popisuje nastalo škodo, in še enkrat velja apel, čim prej pridite v Bučo, pomagajte tem ljudem, preiskava mora iti naprej, in sicer čim prej,\" je opozoril.Žrtvam pobojev sta se v Buči poklonila tudi visoka predstavnika EU-ja, predsednica Evropske komisije Ursula von der Leyen in visoki zunanjepolitični predstavnik Unije Josep Borrell, ki sta obiskala Ukrajino. Množično grobišče sta obiskala skupaj z ukrajinskim premierjem Denisom Šmigaljem in slovaškim predsednikom vlade Eduardom Hegerjem. Voditelji so na grobišče za žrtve položili sveče.Ursula von der Leyen je za vojna grozodejstva v kraju obtožila rusko vojsko. \"Videli smo krut obraz Putinove vojske, videli smo brezobzirnost in hladnokrvnost, s katero so zasedli mesto,\" je dejala.Ukrajincem je prek Twitterja sporočila, da bodo odgovorni za grozodejstva kaznovani. \"Vaš boj je naš boj,\" je še dodala.Ostro je obsodila poboje civilistov v Buči. \"V Buči je bila uničena naša človečnost,\" je dejala ob obisku množičnega grobišča. Obsodila je tudi napad na železniško postajo v Kramatorsku na vzhodu Ukrajine, kar so danes storili tudi številni drugi evropski in svetovni voditelji.\"Mobiliziramo svojo gospodarsko moč, da bo ruski predsednik Vladimir Putin za svoja dejanja plačal zelo, zelo visoko ceno,\" je dejala. Zelenski je dejal, da je sicer osebno hvaležen Ursuli von der Leyen za vse dozdajšnje sankcije EU-ja proti Rusiji, a je dodal, da \"to ni dovolj\".\"Odvzeli so nam veliko stvari, ozemlja, ljudi. Zemljo lahko vzamemo nazaj, ljudi pa nikoli ne bomo mogli vrniti. Za vse to mora Rusija prevzeti odgovornost. Zato vas prosim, da nam pomagate s svojimi sankcijami. Sankcije se morajo samo še zaostriti,\" je dejal.Odzval se je tudi Borrell, ki je prav tako na Twitterju zapisal, da je treba vojne zločine, ki jih je Rusija zagrešila v Buči in drugod, preiskati in preganjati.Moskva medtem še vedno zanika vpletenost v poboj civilistov in trdi, da so posnetki trupel uprizorjeni, vendar za to trditev ni predložila nobenih dokazov.Obvestilo uredništva:Zaradi številnih komentarjev in zagotavljanja čim višjih standardov razprave pod članki o dogajanju v Ukrajini smo se odločili, da komentiranje na portalu rtvslo.si omogočimo pod eno novico. Ne gre za cenzuro ali blokado, temveč za vzdrževanje ravni komunikacije na portalu javne RTV, ki je zavezana k takšnim merilom. Svoje mnenje o dogajanju v Ukrajini lahko ob spoštovanju forumskih pravil MMC RTV SLO izrazite v komentarjih tukaj."
                ]
        text = self.batch(docs[:1])


/usr/local/lib/python3.12/dist-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
INFO:classla:Downloading these customized packages for language: sl (Slovenian)...
| Processor | Package  |
------------------------
| tokenize  | standard |
| pos       | standard |
| lemma     | standard |
| depparse  | standard |
| ner       | standard |
| pretrain  | standard |

INFO:classla:Finished downloading models and saved to /root/classla_resources.


In [ ]:
import pandas as pd
df1 = pd.read_csv('/content/gdrive/MyDrive/slovene_data_2025/processed_slovenian_dataset_1.csv')
df2 = pd.read_csv('/content/gdrive/MyDrive/slovene_data_2025/processed_slovenian_dataset_2.csv')

In [ ]:
merged = pd.concat([df1, df2])


In [ ]:
merged['id'] = range(1, len(merged)+1)

In [ ]:
merged.set_index('id')

,Unnamed: 0,content
id,,
1,0,hrib ločiti tolmin bohinj meter nadmorski viši...
2,1,prioriteta stranka beseda predsednica stranka ...
3,2,stranka nov slovenija NSi napovedovati volilen...
4,3,kolagen beljakovina odgovoren zdrav sklep elas...
5,4,zmaj dvigniti pokal državen prvak sezona preos...
...,...,...
9865,4864,navedba fsbi trojica nameravati sprožiti ekspl...
9866,4865,vojna pojem preteklost resničen začeti leto sk...
9867,4866,kinodvorana tok velik promocijski dejavnost ma...


In [ ]:
merged.drop(merged.columns[merged.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)


In [ ]:
merged = merged.set_index('id')

In [ ]:
merged.to_csv('/content/gdrive/MyDrive/slovene_data_2025/processed_slovenian_dataset_FULL.csv')

In [ ]:
import pandas as pd
import nltk
df = pd.read_csv('/content/gdrive/MyDrive/slovene_data_2025/slovenian_dataset.csv')
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
docs = list(df['content'].astype(str))


In [ ]:
len(docs)

9869

In [ ]:
batched_docs_2 = SloPreProcessing().batch(docs[5000:])



In [ ]:
df.to_csv('/content/gdrive/MyDrive/slovene_data_2025/processed_slovenian_dataset_2.csv')

In [ ]:
final_docs = list(docs)
len(final_docs)

0

In [ ]:
import csv
with open('/content/gdrive/MyDrive/slovene_data_2025/processed_slovenian_dataset_1.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows([docs])

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install --upgrade torch torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
from octis.models.ProdLDA import ProdLDA
from octis.dataset.dataset import Dataset
from octis.models.ETM import ETM
from octis.models.CTM import CTM
from octis.models.NeuralLDA import NeuralLDA
from octis.evaluation_metrics.coherence_metrics import Coherence
from octis.preprocessing.preprocessing import Preprocessing
from octis.optimization.optimizer import Optimizer
from octis.models.LDA import LDA
from skopt.space.space import Real
from octis.evaluation_metrics.similarity_metrics import RBO

NEW_PATH = "/content/gdrive/MyDrive/slovene_data_2025/processed_slovenian_dataset_FULL.csv"
NEW_NEW_PATH = "/content/gdrive/MyDrive/slovene_data_2025/slovenian_dataset.csv"
OLD_PATH = "/content/gdrive/MyDrive/pickles/slo_flattened_docs_1307_2.csv"
class Octis:
    def __init__(self, num_topics, num_layers):
        self.dataset = None
        self.num_topics = num_topics
        self.num_layers = num_layers
        self.corpus = None

    def build_dataset(self):
        preprocessor = Preprocessing(
            vocabulary=None,
            max_features=None,
            remove_punctuation=False,
            lemmatize=False,
            stopword_list=None,
            min_chars=2,
            min_words_docs=1,
            min_df=0.1,
            max_df=0.8,
        )
        # Preprocess
        self.dataset = preprocessor.preprocess_dataset(documents_path=NEW_NEW_PATH)
        self.corpus = self.dataset.get_corpus()
    def build_raw_dataset(self):
        preprocessor = Preprocessing(
            vocabulary=None,
            max_features=None,
            remove_punctuation=False,
            lemmatize=False,
            stopword_list=None,
            min_chars=2,
            min_words_docs=1,
            min_df=0.1,
            max_df=0.8,

        )
        # Preprocess
        self.dataset = preprocessor.preprocess_dataset(documents_path="/content/gdrive/MyDrive/ENG_unprocessed_full.csv")
        print(self.dataset)

    def train_prodlda(self):
        model = ProdLDA(num_topics = self.num_topics, num_layers=self.num_layers)
        return model.train_model(self.dataset)

    def train_etm(self):
        model = ETM(num_topics = self.num_topics,device="gpu", embedding_size=100)
        return model.train_model(self.dataset), model

    def train_cetm(self):
        model = CTM(num_topics = self.num_topics, num_layers=self.num_layers,  inference_type = "zeroshot", bert_model="distiluse-base-multilingual-cased-v2")
        return model.train_model(self.dataset), model

    def train_neuralLDA(self):
        model = NeuralLDA(num_topics = self.num_topics, num_layers=self.num_layers)
        return model.train_model(self.dataset)

    def train_LDA(self):
        model = LDA(num_topics = self.num_topics)
        return model.train_model(self.dataset), model
    def get_coherence(self, model):
        print(model['topics'])
        metric = Coherence(measure="c_v", texts=self.corpus)
        return metric.score(model)

    def get_topic_diversity(self, model):
        from octis.evaluation_metrics.diversity_metrics import TopicDiversity

        metric = TopicDiversity(topk=10) # Initialize metric
        topic_diversity_score = metric.score(model) # Compute score of the metric

        return topic_diversity_score
    def get_similarity(self, model):
        metric = RBO(weight=0.95)
        return metric.score(model)

    def optimize(self, model):
        search_space = {"embedding_size": Real(low=50, high=100)}
        metric = Coherence(measure="c_v", texts=self.corpus)

        # Initialize an optimizer object and start the optimization.
        optimizer=Optimizer()
        optResult=optimizer.optimize(model, self.dataset, metric, search_space, save_path="/content/gdrive/MyDrive/slo_optimization_results_etm", # path to store the results
                                    number_of_call=30, # number of optimization iterations
                                    model_runs=5) # number of runs of the topic model
        #save the results of th optimization in a csv file
        optResult.save_to_csv("/content/gdrive/MyDrive/slo_optimization_results_etm.csv")


In [ ]:
from octis.models.contextualized_topic_models.utils.data_preparation import (
    bert_embeddings_from_list, QuickText)


In [ ]:
import os
import json
import stanza
import gensim
from sklearn.feature_extraction.text import CountVectorizer
import pickle
from nltk.corpus import stopwords
from gensim.models import CoherenceModel
import numpy as np
# from sloPreProcessing import SloPreProcessing
import pickle
import csv


class SloProcessingPipeline:
    def __init__(self):
        # self.sloPre = SloPreProcessing()
        self.docs = None  # Placeholder for processed documents

    def lemmatize(self, text):
        sents_with_lemmas = self.nlp(text)
        return [word.lemma for sent in sents_with_lemmas.sentences for word in sent.words]

    def load_json(self, directory):
        """Load data from multiple JSON files in a directory."""
        combined_texts = []

        for filename in sorted(os.listdir(directory)):
            if filename.endswith('.json'):
                file_path = os.path.join(directory, filename)
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                for item in data:
                    try:
                        #combined_text = f"{data[item]['title']} {data[item]['subtitle']} {data[item]['slug']}"
                        combined_text = data[item]['content']
                        combined_texts.append(combined_text)

                    except:
                        print(f"failed reading {item}")

        return combined_texts

    def batch(self, docs):
        """Process a list of documents with the Stanza pipeline."""
        """
        # batched = [stanza.Document([], text=d.lower()) for d in docs]

        parsed = self.nlp(docs)

        self.docs = [self.sloPre.lemmatize(doc) for doc in parsed]
        """
        self.docs = self.sloPre.batch(docs)


    def get_flattened_docs(self):
        """Flatten the lemmatized documents into a list of strings."""
        if self.docs is None:
            raise EnvironmentError("No documents available. Run batch method first.")
        return [' '.join(doc) for doc in self.docs]
    def write_file(self, content):
        with open("/content/gdrive/MyDrive/pickles/slo_flattened_docs_1307_2.csv", mode='w', newline="") as file:
            writer = csv.writer(file)
            for sentence in content:
                writer.writerow([sentence])  # Each sentence goes in a new row


    def get_octis(self):
        oc = Octis(num_topics=15, num_layers = 4)
        oc.build_dataset()

        return oc

    def main(self):
        load = True

        """if not load:
        # Load and process documents from a directory of JSON files
            docs = self.load_json(directory)
            self.batch(docs)
            flattened_docs = self.get_flattened_docs()
            print(len(flattened_docs))
            with open('pickles/slo_flattened_docs_1407.pkl', 'wb') as f:
                pickle.dump(self.docs, f)
        else:
            flatteneddocs = open('/content/gdrive/MyDrive/pickles/slo_flattened_docs_1307_2.pkl', 'rb')
            self.docs = pickle.load(flatteneddocs)
            flattened_docs = self.get_flattened_docs()
        """

        #self.flattened_docs = flattened_docs
        oc = Octis(num_topics=30, num_layers = 3)
        # corpus - flattened docs
        # vocab - dictionary
        oc.build_dataset()

        lda = oc.train_LDA()

        lda_metric = oc.get_coherence(lda)

        etm,_ = oc.train_etm()
        prodlda = oc.train_prodlda()
        nlda = oc.train_neuralLDA()
        cetm = oc.train_cetm()

        print(f"lda topics {lda['topics']}")
        print(f"etm topics {etm['topics']}")
        print(f"prodlda topics {prodlda['topics']}")
        print(f"nlda topics {nlda['topics']}")
        print(f"cetm topics {cetm['topics']}")

        etm_metric = oc.get_coherence(etm)
        prod_lda_metric = oc.get_coherence(prodlda)
        nlda_metric = oc.get_coherence(nlda)
        cetm_metric = oc.get_coherence(cetm)

        print(f"coherence for lda: {lda_metric}, etm: {etm_metric}, prod_lda: {prod_lda_metric}, nlda: {nlda_metric}, cetm: {cetm_metric}")

        lda_div_metric = oc.get_topic_diversity(lda)
        etm_div_metric = oc.get_topic_diversity(etm)
        prod_lda_div_metric = oc.get_topic_diversity(prodlda)
        nlda_div_metric = oc.get_topic_diversity(nlda)
        cetm_div_metric = oc.get_topic_diversity(cetm)

        print(f"diversity for lda: {lda_div_metric}, etm: {etm_div_metric}, prod_lda: {prod_lda_div_metric}, nlda: {nlda_div_metric}, cetm: {cetm_div_metric}")

        lda_sim_metric = oc.get_similarity(lda)
        etm_sim_metric = oc.get_similarity(etm)
        prod_lda_sim_metric = oc.get_similarity(prodlda)
        nlda_sim_metric = oc.get_similarity(nlda)
        cetm_sim_metric = oc.get_similarity(cetm)

        print(f"diversity for lda: {lda_sim_metric}, etm: {etm_sim_metric}, prod_lda: {prod_lda_sim_metric}, nlda: {nlda_sim_metric}, cetm: {cetm_sim_metric}")






In [ ]:



    # Path to your directory containing JSON files
    s = SloProcessingPipeline()
    results = s.main()


100%|██████████| 7526/7526 [00:03<00:00, 1986.96it/s]


created vocab
494


TypeError: tuple indices must be integers or slices, not str

In [ ]:
s = SloProcessingPipeline()
octis = s.get_octis()

100%|██████████| 15577/15577 [00:07<00:00, 2064.97it/s]


created vocab
239


In [ ]:
lda_trained = octis.train_LDA()[0]
coherence = octis.get_coherence(lda_trained)
diversity = octis.get_topic_diversity(lda_trained)
print(f"Coh: {coherence}, div: {diversity}")

[['za', 'in', 'so', 'tudi', 'je', 'ki', 'na', 'da', 'se', 'ter'], ['je', 'in', 'za', 'da', 'na', 'so', 'se', 'ki', 'kot', 'tudi'], ['je', 'na', 'in', 'se', 'za', 'so', 'po', 'pa', 'ki', 'da'], ['je', 'da', 'in', 'na', 'so', 'se', 'ki', 'za', 'bi', 'več'], ['je', 'da', 'in', 'na', 'se', 'so', 'to', 'za', 'bi', 'ki'], ['so', 'in', 'je', 'na', 'se', 'da', 'za', 'ki', 'tudi', 'po'], ['je', 'in', 'na', 'so', 'ki', 'da', 'za', 'pa', 'tudi', 'se'], ['da', 'in', 'je', 'se', 'ki', 'tudi', 'na', 'to', 'za', 'ne'], ['na', 'in', 'je', 'za', 'bo', 'so', 'ki', 'se', 'pa', 'po'], ['je', 'da', 'in', 'za', 'na', 'bi', 'se', 'bo', 'ki', 'ne'], ['je', 'in', 'da', 'se', 'na', 'sem', 'ki', 'ne', 'za', 'to'], ['je', 'in', 'da', 'so', 'na', 'za', 'ki', 'se', 'po', 'še'], ['je', 'na', 'se', 'in', 'za', 'tudi', 'da', 'ki', 'pa', 'lahko'], ['je', 'in', 'se', 'na', 'ki', 'za', 'pa', 'so', 'kot', 'tudi'], ['slovenija', 'in', 'smo', 'ob', 'da', 'se', 'so', 'ker', 'kot', 'je']]
Coh: 0.32162787488285033, div: 0.1666

In [ ]:
prodlda_trained = octis.train_prodlda()
coherence = octis.get_coherence(prodlda_trained)
diversity = octis.get_topic_diversity(prodlda_trained)
print(f"Coh: {coherence}, div: {diversity}")

Epoch: [1/100]	Samples: [6880/688000]	Train Loss: 904.5284713390262	Time: 0:00:00.720087
Epoch: [1/100]	Samples: [1475/147500]	Validation Loss: 866.3230946272511	Time: 0:00:00.056871
Epoch: [2/100]	Samples: [13760/688000]	Train Loss: 870.099522506359	Time: 0:00:00.526266
Epoch: [2/100]	Samples: [1475/147500]	Validation Loss: 842.6003411347988	Time: 0:00:00.056303
Epoch: [3/100]	Samples: [20640/688000]	Train Loss: 858.4555851426236	Time: 0:00:00.545603
Epoch: [3/100]	Samples: [1475/147500]	Validation Loss: 834.5268635791844	Time: 0:00:00.056101
Epoch: [4/100]	Samples: [27520/688000]	Train Loss: 848.2259916083757	Time: 0:00:00.539463
Epoch: [4/100]	Samples: [1475/147500]	Validation Loss: 824.3450105932203	Time: 0:00:00.056919
Epoch: [5/100]	Samples: [34400/688000]	Train Loss: 839.4670506676962	Time: 0:00:00.532091
Epoch: [5/100]	Samples: [1475/147500]	Validation Loss: 818.4071330442267	Time: 0:00:00.057833
Epoch: [6/100]	Samples: [41280/688000]	Train Loss: 833.8903513353924	Time: 0:00:00

In [ ]:
cetm_trained = octis.train_cetm()[0]
coherence = octis.get_coherence(cetm_trained)
diversity = octis.get_topic_diversity(cetm_trained)
print(f"Coh: {coherence}, div: {diversity}")

Exception: Wait! BoW and Contextual Embeddings have different sizes! You might want to check if the BoW preparation method has removed some documents. 